# X-VC — преобразование голоса

Этот блокнот запускает официальный X-VC с простым Gradio-интерфейсом.

Перед запуском в Colab выберите среду выполнения с GPU. Для первого запуска понадобятся несколько гигабайт загрузок: сам checkpoint X-VC около 5 ГБ, плюс вспомогательные модели.

Запустите первую ячейку один раз. Затем запустите вторую. Она запускает Gradio в фоне, ждёт публичную ссылку и завершается. Когда интерфейс будет готов, ссылка появится отдельной строкой и отдельной ссылкой «Открыть X-VC». Если Gradio не запустится, ячейка покажет последние строки ошибки вместо бесконечного ожидания.


In [ ]:
import os, subprocess
assert subprocess.run(['nvidia-smi'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode == 0, 'Включите GPU в настройках среды выполнения Colab.'
subprocess.run(['rm', '-rf', '/content/audio-restoration-colab'], check=True)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'agent/x-vc-colab', 'https://github.com/egor125552/audio-restoration-colab.git', '/content/audio-restoration-colab'], check=True)
subprocess.run(['bash', '/content/audio-restoration-colab/x_vc_colab/install_xvc.sh', '/content/x-vc'], check=True)
print('X-VC установлена. Теперь запускайте следующую ячейку.')


In [ ]:
import re, subprocess
from IPython.display import Markdown, display

result = subprocess.run(
    [
        'python3', '-u',
        '/content/audio-restoration-colab/x_vc_colab/launch_colab.py',
        '--python', '/content/x-vc/.venv/bin/python',
        '--port', '7860',
        '--timeout', '90',
        '--log', '/content/xvc-gradio.log',
        '--pid-file', '/content/xvc-gradio.pid',
    ],
    text=True,
    capture_output=True,
)

if result.stdout:
    print(result.stdout, end='')
if result.returncode != 0:
    if result.stderr:
        print(result.stderr, end='')
    raise RuntimeError('Не удалось запустить Gradio. Ошибка напечатана выше.')

match = re.search(r'XVC_PUBLIC_URL=(https://[^\s]+)', result.stdout)
if not match:
    raise RuntimeError('Launcher завершился без публичной ссылки.')
public_url = match.group(1)
print('Публичная ссылка X-VC:')
print(public_url)
display(Markdown(f'[Открыть X-VC]({public_url})'))


In [ ]:
import os, signal
from pathlib import Path
pid_path = Path('/content/xvc-gradio.pid')
if pid_path.exists():
    try:
        os.kill(int(pid_path.read_text().strip()), signal.SIGTERM)
        pid_path.unlink(missing_ok=True)
        print('Интерфейс X-VC остановлен.')
    except ProcessLookupError:
        pid_path.unlink(missing_ok=True)
        print('Интерфейс уже остановлен.')
else:
    print('Запущенный интерфейс не найден.')
